# R309 - Socket - Interface de connexion

<br/>
<div style="border:solid 1px green; color:green">
    <p> Une (ou un) <code>socket</code> réseau est une <code>interface de connexion</code> servant de point d'extrémité pour la <code>transmission et la réception de données</code>.</p>
    <p> Très classiquement dans les réseaux IP, elle est liée à un couple <code>(Adresse IP, Numéro de port)</code> (<code>IP/TCP</code> ou <code>IP/UDP</code>).</p>
</div> 

Pour créer une socket, il existe une méthode appelée <code>socket</code> du <code>module socket</code>. 

Elle prend au moins en arguments :
- Une famille de socket (<code>socket.AF_INET</code> pour IPv4, ou socket.AF_INET6 pour IPv6, ou ...)
- Un type de socket (<code>socket.SOCK_STREAM</code> pour TCP, ou <code>socket.SOCK_DGRAM</code> pour UDP, ou ...)

Plus détails dans la <a href="https://docs.python.org/fr/3/library/socket.html#module-socket">documentation sur le module socket</a>

Nous allons utiliser dans ce cours des sockets IPv4 / TCP.


Les <strong>principales méthodes</strong> utilisables sur cet <strong>objet socket</strong> sont :

<code><strong>bind()</strong>, <strong>listen()</strong> and <strong>accept()</strong></code> servant <strong>côté  <code>serveur</code></strong>.

<code><strong>connect()</strong></code> servant <strong>côté  <code>client</code></strong>.

<code><strong>send()</strong> et <strong>recv()</strong></code> servant  <strong><code>pour les deux</code></strong>.

## Serveur mininimal IPv4/TCP demandant à un client de lui envoyer un message

Côté Serveur : <code>socket_serveur_min.py</code> (Choix : nom des variables en français)

In [ ]:
# Définition d'un serveur minimal
# Demande à un client de lui envoyer un message

import socket

# Adresse IP et numéro de port du Serveur
SERVEUR_HOTE = "127.0.0.1" # ici on prend localhost 
SERVEUR_PORT = 52300       # choix au dessus de 1024
  
# 1) Création d'un ou une socket IPv4 / TCP :  AF_INET = IPv4 et SOCK_STREAM = TCP
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    # 2) Liaison du socket à une adresse précise : 
    s.bind((SERVEUR_HOTE, SERVEUR_PORT)) # attention le paramètre est un tuple

    # 3) Attente de la requête de connexion d'un client : 
    print("[Serveur] Serveur prêt, en attente de requêtes ...") 
    s.listen(1)  # Accepte 1 client maximum
  
    # 4) Etablissement de la connexion : 
    connexion, adresse = s.accept()
    print(f"[Serveur] Client {adresse[0]}:{adresse[1]} vient de se connecter.")
  
    # 5) Dialogue avec le client : envoi de la question et attente de la réponse 
    connexion.send("Vous êtes connecté au serveur. Quel est votre message ? ".encode("utf8")) 
    print("[Serveur] Message du client :", connexion.recv(1024).decode("utf8"))
  
    # 6) Fermeture de la connexion : 
    print("[Serveur] Fin du programme.")

    # La clause with ferme automatiquement la connexion

Côté Client : <code>socket_client_min.py</code> (Choix nom des variables en français)

In [ ]:
# Définition d'un client minimum 
# Ce client envoi un message au serveur
  
import socket

# Adresse IP et Numéro de port du Serveur à l'écoute
SERVEUR_HOTE = "127.0.0.1"
SERVEUR_PORT = 52300 

# 1) Création d'un ou une socket IPv4 / TCP :  AF_INET = IPv4 et SOCK_STREAM = TCP
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    # 2) Envoi d'une requête de connexion au serveur : 
    print(f"[Client] Connexion au Serveur {SERVEUR_HOTE}:{SERVEUR_PORT}")
    s.connect((SERVEUR_HOTE, SERVEUR_PORT)) # attention c'est un tuple
    print("[Client] Connexion établie avec Serveur.") 
    
    # 3) Dialogue avec le serveur 
    print(s.recv(1024).decode("utf8"))  # Réception d'un message issu du serveur
    s.send(input().encode("utf8"))      # Envoi d'un message issu du client
    print("[Client] Message envoyé au Serveur.")
    
    # 4) Fermeture de la connexion : 
    print("[Client] Fin du programme.")
    
    # La clause with ferme automatiquement la connexion

En résumé voici les méthodes utilisées côté Serveur et côté Client
![socket](socket.png "Socket")

## Le serveur reste actif et la communication avec un client s'arrête quand il envoie une chaîne stockée dans END_MSG (on met "q") 

Côté Serveur : <code>socket_server.py</code> (Choix : nom des variables en <strong>anglais</strong>)

En particulier :
* Serveur => Server
* Connexion => Connection
* Adresse => Address
* Hôte => Host

In [ ]:
# Serveur

import socket

# IP Address and port number for the SERVER
SERVER_HOST = "127.0.0.1" # localhost 
SERVER_PORT = 52300       # greater than 1024
ENDING_MSG = "q"
  
# 1) Creation of an IPv4/TCP socket:  AF_INET = IPv4 / SOCK_STREAM = TCP
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    # 2) Binding the socket
    s.bind((SERVER_HOST, SERVER_PORT)) # Be careful than the paramater is a tuple

    # 3) Waiting for a client 
    print("[Serveur] Serveur prêt, en attente d'un client.") 
    s.listen(1)  # Accept 1 client at most
  
    while True:
        # 4) Establishing the connection with a client
        connection, address = s.accept()
        print(f"[Serveur] Client {address[0]}:{address[1]} vient de se connecter.") 
  
        # 5) Discussing with the client while it sends not ENDING_MSG
        server_msg = f"Vous êtes connecté au serveur. Quel est votre message ?"+\
                f" ({ENDING_MSG} pour quitter)"
        connection.send(server_msg.encode("utf8"))
        client_msg = ""
        while client_msg != ENDING_MSG:
            client_msg = connection.recv(1024).decode("utf8")
            print(f"[Client] {client_msg}")
            server_msg = input("[Serveur] ")
            connection.send(server_msg.encode("utf8"))
        print(f"[Serveur] Client déconnecté. En attente d'un nouveau client.")
  

Côté Client : <code>socket_client.py</code> (Choix : nom des variables en <strong>anglais</strong>)

In [ ]:
# Client
  
import socket

# IP Address and port number for the SERVER
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 52300 
ENDING_MSG = "q"

# 1) Creation of an IPv4/TCP socket:  AF_INET = IPv4 / SOCK_STREAM = TCP
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    # 2) Connecting the server
    print(f"[Client] Connexion au Serveur {SERVER_HOST}:{SERVER_PORT}")
    s.connect((SERVER_HOST, SERVER_PORT)) # Watch out the tuple!
    print(f"[Client] Connexion établie avec {SERVER_HOST}:{SERVER_PORT}") 
    
    # 3) Dialogue avec le serveur 
    client_msg = ""
    while client_msg != ENDING_MSG:
        server_msg = s.recv(1024).decode('utf8')
        print(f"[Serveur] {server_msg}")
        client_msg = input("[Client] ")
        s.send(client_msg.encode("utf8"))
    
    # 4) Fermeture de la connexion : 
    print("[Client] Fin connexion.")
    
    # La clause with ferme automatiquement la connexion


## Serveur avec plusieurs clients (un thread par client)

L'idée est maintenant d'avoir un <code>serveur gérant un tchat de clients/utilisateurs</code>.

<code>Côté serveur</code>, on choisit de lancer <code>un thread par client</code> qui se connecte pour gérer les messages de ces derniers.

<code>Côté client</code>, comme on doit recevoir les messages de tous les autres clients tout en faisant en sorte que le client puisse parler (méthode input()), on choisit de lancer <code>un thread pour gérer les messages du serveur</code> (il diffuse/broadcast les messages des autres clients connectés).

<code>server.py</code>

In [ ]:
import socket
import threading

SERVER_HOST = "127.0.0.1"
SERVER_PORT = 52300 
ENDING_MSG = "q"

# Client's connections
connections = []

def handle_client_msg(connection: socket.socket, address: str) -> None:
    """
        Handle messages for one client
    """
    client_msg = ""
    while client_msg != ENDING_MSG:
        client_msg = connection.recv(1024).decode("utf8")
        print(f"[Client {address[0]}:{address[1]}] {client_msg}")
        # Message to broadcast to other clients.
        msg_to_send = f"[Client {address[0]}:{address[1]}] {client_msg}"
        broadcast(msg_to_send, connection)   
    # Client has left
    remove(connection, address)
    

def remove(connection: socket.socket, address: str) -> None:
    """
        Remove a client connection 
        (He/she has typed ENDING_MSG)
    """
    connections.remove(connection) 
    print(f"[Serveur] Client {address[0]}:{address[1]} vient de se déconnecter.")

def broadcast(message: str, connection: socket.socket) -> None:
    """
        Broadcast a client message to all other clients connected to the server
    """
    for client_connection in connections:
        if client_connection != connection:
            client_connection.send(message.encode("utf8"))

def server() -> None:
    """
        Main process: receive client's connections and
        start a new thread for each one to handle their messages
    """
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind((SERVER_HOST, SERVER_PORT))
        s.listen(4)  # Accept 4 clients at most
        print("[Serveur] Serveur prêt.")        
        while True:
            connection, address = s.accept()
            print(f"[Serveur] Client {address[0]}:{address[1]} vient de se connecter.")
            connections.append(connection)
            threading.Thread(target=handle_client_msg, args=[connection, address]).start()

if __name__ == "__main__":
    server()

<code>client.py</code>

In [ ]:
import socket
import threading

SERVER_HOST = "127.0.0.1"
SERVER_PORT = 52300 
ENDING_MSG = "q"

def handle_server_msg(connection: socket.socket, client_prompt: str) -> None:
    """
        Receive and print messages sent by the server 
        while the client is connected
    """
    while True:
        try:
            server_msg = connection.recv(1024).decode("utf8")
            print(f"\n{server_msg}\n{client_prompt}", end="")
        except:
            # Client has left (ENDING_MSG has been typed)
            break

def client() -> None:
    """
        Main process: start a client and handle messages
    """
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        print(f"** Connexion au Serveur {SERVER_HOST}:{SERVER_PORT} **")
        s.connect((SERVER_HOST, SERVER_PORT)) # Watch out the tuple!
        print(f"** Connexion établie (Tapez {ENDING_MSG} pour quitter)**")
        client_prompt = f"[Client {s.getsockname()[0]}:{s.getsockname()[1]} (Vous)] "
        # Create a thread to handle messages sent by the server
        threading.Thread(target=handle_server_msg, args=[s, client_prompt]).start()
        # Deal the client's inputs
        client_msg = ""
        while client_msg != ENDING_MSG:
            print(client_prompt, end="")
            client_msg = input()
            s.send(client_msg.encode("utf8"))
    
    # 4) Fermeture de la connexion : 
    print("** Fin connexion ** Au revoir **")

if __name__ == "__main__":
    client()